# Face-Car — CNN de olhos e boca

Este notebook treina duas CNNs independentes com dados reais: olhos (`aberto`/`fechado`) e boca (`bocejo`/`sem_bocejo`). Não baixa datasets automaticamente e não cria rótulos.

O notebook original do professor, `Aula10_Colab_CIFAR10.ipynb`, é preservado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q tensorflow opencv-python-headless scikit-learn matplotlib

In [ ]:
import sys, subprocess
from pathlib import Path
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU disponível:', bool(tf.config.list_physical_devices('GPU')))
REPO = Path('/content/face-car')
if not (REPO / 'treinamento' / 'preparar_regioes.py').exists():
    subprocess.run(['git', 'clone', '--branch', 'feature/treinamento-avaliacao', '--single-branch', 'https://github.com/dario-bastos-dev/face-car.git', str(REPO)], check=True)

## Arquivos no Google Drive

Coloque manualmente os dados reais. O CEW deve ser organizado com patches dos olhos e manifesto `manifesto_cew.csv`. O YawDD deve ter vídeos e manifesto `manifesto_yawdd.csv`. Coloque também `face_landmarker.task` no Drive para o recorte da boca.

In [ ]:
DATASETS = Path('/content/drive/MyDrive/Face-Car/datasets')
CEW_ROOT = DATASETS / 'cew'
YAWDD_ROOT = DATASETS / 'yawdd'
MODEL_MP = DATASETS / 'face_landmarker.task'
OUT = Path('/content/drive/MyDrive/Face-Car/resultados_cnn')
OUT.mkdir(parents=True, exist_ok=True)
print('CEW:', CEW_ROOT)
print('YawDD:', YAWDD_ROOT)
print('MediaPipe:', MODEL_MP)

## 1. Preparar olhos com CEW

Manifesto esperado: `image_path,label,person_id`, com `label` igual a `aberto` ou `fechado`. O `person_id` mantém a pessoa inteira em um único split. Os patches CEW já são a região dos olhos; não serão recortados novamente.

In [ ]:
MANIFESTO_CEW = CEW_ROOT / 'manifesto_cew.csv'
DADOS_OLHOS = DATASETS / 'olhos_processado'
if MANIFESTO_CEW.exists():
    subprocess.run([sys.executable, str(REPO / 'treinamento' / 'preparar_regioes.py'), '--region', 'olhos', '--manifest', str(MANIFESTO_CEW), '--input-root', str(CEW_ROOT), '--input-roi', '--output-dir', str(DADOS_OLHOS), '--image-size', '96'], check=True)
else:
    print('Envie o manifesto CEW e execute esta célula.')

## 2. Preparar boca com YawDD + MediaPipe

Manifesto esperado: `video_path,label,driver_id`, com `label` definido pela anotação real do vídeo ou intervalo. A preparação só salva frames quando o MediaPipe encontra o rosto e a boca.

In [ ]:
MANIFESTO_YAWDD = YAWDD_ROOT / 'manifesto_yawdd.csv'
DADOS_BOCA = DATASETS / 'boca_processado'
if MANIFESTO_YAWDD.exists() and MODEL_MP.exists():
    subprocess.run([sys.executable, str(REPO / 'treinamento' / 'preparar_regioes.py'), '--region', 'boca', '--manifest', str(MANIFESTO_YAWDD), '--input-root', str(YAWDD_ROOT), '--mediapipe-model', str(MODEL_MP), '--output-dir', str(DADOS_BOCA), '--frame-stride', '5', '--image-size', '96'], check=True)
else:
    print('Envie o manifesto YawDD e face_landmarker.task e execute esta célula.')

## 3. Treinar e avaliar CNN dos olhos

Modelo A: até 15 épocas. Modelo B: augmentation, Dropout e EarlyStopping até 30 épocas.

In [ ]:
OUT_OLHOS = OUT / 'olhos'
subprocess.run([sys.executable, str(REPO / 'treinamento' / 'treinar_regioes.py'), '--region', 'olhos', '--data-dir', str(DADOS_OLHOS), '--output-dir', str(OUT_OLHOS)], check=True)
subprocess.run([sys.executable, str(REPO / 'treinamento' / 'avaliar_regioes.py'), '--region', 'olhos', '--data-dir', str(DADOS_OLHOS), '--models-dir', str(OUT_OLHOS)], check=True)

## 4. Treinar e avaliar CNN da boca

A CNN usa somente os recortes da boca; não usa o rosto inteiro como atalho.

In [ ]:
OUT_BOCA = OUT / 'boca'
subprocess.run([sys.executable, str(REPO / 'treinamento' / 'treinar_regioes.py'), '--region', 'boca', '--data-dir', str(DADOS_BOCA), '--output-dir', str(OUT_BOCA)], check=True)
subprocess.run([sys.executable, str(REPO / 'treinamento' / 'avaliar_regioes.py'), '--region', 'boca', '--data-dir', str(DADOS_BOCA), '--models-dir', str(OUT_BOCA)], check=True)

## Saídas e etapa temporal futura

Cada pasta de região salva `modelo_a.keras`, `modelo_b.keras`, `melhor_modelo.keras`, `metricas.json` e matrizes de confusão. A análise temporal inicial está em `treinamento/analise_temporal.py` e usará EAR, abertura da boca, pitch, velocidade do pitch e confiança para calcular piscadas, PERCLOS, bocejos e quedas da cabeça.

Nenhum resultado deve ser interpretado antes de confirmar a origem dos rótulos e a separação por pessoa/motorista.